In [1]:
import json
import shutil
from pathlib import Path

PROJECT = Path.home() / "projects/knee-oa-classification"
HF_DIR = PROJECT / "hf_model"
HF_DIR.mkdir(exist_ok=True)

for name in ["exp1_augmentation", "exp3_aug_wider"]:
    shutil.copy(PROJECT / "models" / f"{name}.keras", HF_DIR / f"{name}.keras")
shutil.copy(PROJECT / "src/preprocessing.py", HF_DIR / "preprocessing.py")
shutil.copy(PROJECT / "MODEL_CARD.md", HF_DIR / "README.md")       # Hugging Face shows README.md as the card

config = {
    "version": "1.0",
    "members": ["exp1_augmentation.keras", "exp3_aug_wider.keras"],
    "input": {"shape": [224, 224], "channels": 1, "dtype": "uint8", "scale": "divide by 255"},
    "decision_rule": "argmax(mean_softmax / train_prior)",
    "train_counts_by_grade": [2286, 1046, 1516, 757, 173],
    "grade_names": ["Healthy", "Doubtful", "Minimal", "Moderate", "Severe"],
    "gradcam_layer": "block4_relu2",
}
(HF_DIR / "pipeline_config.json").write_text(json.dumps(config, indent=2))

for f in sorted(HF_DIR.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:6.2f} MB")

README.md                      0.01 MB
exp1_augmentation.keras       14.23 MB
exp3_aug_wider.keras          31.81 MB
pipeline_config.json           0.00 MB
preprocessing.py               0.00 MB


In [2]:
from huggingface_hub import HfApi

api = HfApi()
user = api.whoami()["name"]
repo_id = f"{user}/knee-oa-kl-grading"

api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path=HF_DIR, repo_id=repo_id, repo_type="model",
                  commit_message="v1.0: ensemble of two custom CNNs + preprocessing")
print(f"Uploaded: https://huggingface.co/{repo_id}")

/home/afzal/miniconda3/envs/knee-oa/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Uploaded: https://huggingface.co/afzal2003/knee-oa-kl-grading


In [3]:
import sys
import numpy as np
import keras
from huggingface_hub import hf_hub_download

sys.path.insert(0, str(PROJECT / "src"))
import data
import evaluation as ev

models_hub = [keras.models.load_model(hf_hub_download(repo_id, f)) for f in config["members"]]
X_val, y_val = data.load_split("val")
val_ds = data.make_dataset(X_val, y_val, batch_size=64)

probs = np.mean([m.predict(val_ds, verbose=0) for m in models_hub], axis=0)
prior = np.array(config["train_counts_by_grade"]) / sum(config["train_counts_by_grade"])
m = ev.compute_metrics(y_val, (probs / prior).argmax(axis=1))
print(f"Models downloaded from the Hub: validation balanced accuracy {m['balanced_accuracy']:.4f} (expect 0.6467)")

I0000 00:00:1791140974.348052  100640 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791140974.626384  100640 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791140976.250218  100640 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791140982.912250  100640 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB mem

Models downloaded from the Hub: validation balanced accuracy 0.6467 (expect 0.6467)


In [4]:
%%writefile /home/afzal/projects/knee-oa-classification/space/app.py
"""Knee osteoarthritis KL-grade demo. Models are downloaded from the Hugging Face Hub."""
import json
import os
from datetime import datetime
from pathlib import Path

import cv2
import gradio as gr
import keras
import numpy as np
from huggingface_hub import hf_hub_download

import gradcam as gc
import preprocessing as pp

REPO_ID = os.environ.get("MODEL_REPO", "afzal2003/knee-oa-kl-grading")
GITHUB_URL = "https://github.com/YOUR_USERNAME/knee-oa-classification"
HERE = Path(__file__).parent
ASSETS, EXAMPLES_DIR = HERE / "assets", HERE / "examples"

# ---------- Load the frozen pipeline from the Hub ----------
config = json.loads(Path(hf_hub_download(REPO_ID, "pipeline_config.json")).read_text())
MODELS = [keras.models.load_model(hf_hub_download(REPO_ID, f)) for f in config["members"]]
GRAD_MODELS = gc.build_gradcam_models(MODELS, layers=(config["gradcam_layer"],))
COUNTS = np.array(config["train_counts_by_grade"])
PRIOR = COUNTS / COUNTS.sum()
NAMES = config["grade_names"]
DESCRIPTIONS = [
    "No radiographic features of osteoarthritis.",
    "Doubtful joint space narrowing with possible osteophytic lipping.",
    "Definite osteophytes and possible joint space narrowing.",
    "Multiple osteophytes, definite joint space narrowing and mild sclerosis.",
    "Large osteophytes, marked joint space narrowing and severe sclerosis.",
]


# ---------- Prediction ----------
def to_model_input(image):
    img = np.asarray(image)
    if img.ndim == 3:
        img = cv2.cvtColor(img[..., :3], cv2.COLOR_RGB2GRAY)
    img = img.astype(np.uint8)
    if img.shape != (224, 224):
        img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)
    return img


def grade_card(grade, top):
    md = f"### Predicted KL grade {grade}: {NAMES[grade]}\n{DESCRIPTIONS[grade]}"
    if grade != top:
        md += (f"\n\n> The model's highest raw probability was grade {top} ({NAMES[top]}). "
               f"The final grade corrects for how rare grade {grade} is in the training data, "
               "which improves performance across all grades. See the **About** tab.")
    return md


def probability_bars(probs, grade):
    rows = []
    for g, p in enumerate(probs):
        weight = "600" if g == grade else "400"
        rows.append(
            f"<div style='display:flex;align-items:center;gap:10px;margin:5px 0;font-weight:{weight}'>"
            f"<span style='width:120px'>{g} · {NAMES[g]}</span>"
            f"<div style='flex:1;height:10px;border-radius:5px;background:var(--border-color-primary)'>"
            f"<div style='width:{p * 100:.1f}%;height:10px;border-radius:5px;background:var(--color-accent)'></div></div>"
            f"<span style='width:44px;text-align:right'>{p:.0%}</span></div>")
    return "<div><b>Model probabilities</b>" + "".join(rows) + "</div>"


def analyze(image, show_cam, history, progress=gr.Progress()):
    if image is None:
        raise gr.Error("Upload a knee X-ray first.")
    try:
        progress(0.1, desc="Preprocessing the X-ray")
        x_pre = pp.preprocess(to_model_input(image))
    except Exception:
        raise gr.Error("This image couldn't be processed. Try another PNG or JPG file.")

    progress(0.4, desc="Running both models")
    x = x_pre[None, ..., None].astype("float32") / 255.0
    probs = np.mean([m(x, training=False).numpy()[0] for m in MODELS], axis=0)
    grade, top = int((probs / PRIOR).argmax()), int(probs.argmax())

    cam_img = None
    if show_cam:
        progress(0.7, desc="Computing the heatmap")
        cam_img = gc.overlay(x_pre, gc.ensemble_gradcam(GRAD_MODELS, x_pre, grade))

    progress(1.0, desc="Done")
    history = (history or []) + [[datetime.now().strftime("%H:%M:%S"),
                                  f"{grade} · {NAMES[grade]}", f"{probs[grade]:.0%}"]]
    return grade_card(grade, top), probability_bars(probs, grade), x_pre, cam_img, history, history[::-1]


# ---------- Static content ----------
INSIGHTS_MD = """
## Test-set performance
Evaluated once on **1,656 held-out X-rays**, after the pipeline was frozen. Intervals are 95% bootstrap confidence intervals.

| Metric | Value | 95% CI |
|---|---|---|
| Balanced accuracy | **0.644** | 0.621 – 0.666 |
| Quadratic weighted kappa | **0.753** | 0.730 – 0.775 |
| Accuracy | 0.542 | |

| Grade | 0 Healthy | 1 Doubtful | 2 Minimal | 3 Moderate | 4 Severe |
|---|---|---|---|---|---|
| Recall | 0.44 | 0.44 | 0.58 | 0.80 | 0.96 |
| Precision | 0.80 | 0.24 | 0.59 | 0.71 | 0.69 |

Severe and moderate OA are recognized well. The hard part is separating grades 0, 1 and 2: the model flags 46% of healthy knees as grade 1 ("doubtful"). Almost all errors are to an adjacent grade.

## Training data: grade distribution
| Grade | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| Training images | 2,286 (40%) | 1,046 (18%) | 1,516 (26%) | 757 (13%) | 173 (3%) |

A model that always predicted "healthy" would score 40% accuracy but only **20% balanced accuracy**, which is why balanced accuracy is the target metric.

## How the final pipeline was chosen (validation results)
| Run | Balanced acc. | QWK |
|---|---|---|
| Always predict grade 0 | 0.200 | 0.000 |
| Baseline CNN | 0.591 | 0.556 |
| + Augmentation | 0.613 | 0.721 |
| + Class weights | 0.603 | 0.723 |
| Wider network | 0.615 | 0.713 |
| Ordinal (CORAL) head | 0.555 | 0.725 |
| Raw images, no preprocessing | 0.589 | 0.693 |
| **Ensemble + prior adjustment (final)** | **0.647** | **0.712** |

## Explainability checks
- **70%** of the Grad-CAM signal falls on the central joint band, which covers 31% of the image.
- **Deletion test:** erasing the regions an explanation ranks most important collapses the model's confidence far faster than erasing random regions (area under curve 0.154–0.170 vs 0.422).
- **Sanity check:** heatmaps from the trained model are unrelated to those of an untrained copy (Spearman −0.03), so they reflect what the model learned.
"""

ABOUT_MD = f"""
## How it works
1. **Preprocessing:** the image is converted to 224×224 grayscale. Inverted X-rays are detected and flipped back, contrast is stretched using the central joint region, and CLAHE enhances local edges.
2. **Two custom CNNs**, trained from scratch, each output a probability for grades 0–4. Their probabilities are averaged.
3. **Prior adjustment:** each probability is divided by how common that grade was in training, removing the model's lean toward common grades. This is why the predicted grade is occasionally not the tallest bar.
4. **Grad-CAM** highlights the regions that most influenced the predicted grade.

## Expected input
Front-view knee X-rays **cropped to the joint**, like the training images. The model outputs a grade for any image, but results for other images (other views, other body parts, photos) are meaningless.

## Limitations
- Trained and tested on a single source (the Osteoarthritis Initiative). Performance elsewhere is unknown.
- Grade 1 ("doubtful") is not reliably separated from grades 0 and 2.
- The public data has no patient identifiers, so a patient's two knees may appear in different splits, which could make results slightly optimistic.
- Heatmaps are validated for faithfulness but not against expert lesion annotations.

## Links
- Code and full write-up: [{GITHUB_URL}]({GITHUB_URL})
- Model card: [huggingface.co/{REPO_ID}](https://huggingface.co/{REPO_ID})

## Data attribution
Chen, Pingjun (2018). *Knee Osteoarthritis Severity Grading Dataset*. Mendeley Data, V1. DOI: [10.17632/56rmx5bjcr.1](https://doi.org/10.17632/56rmx5bjcr.1). Licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Organized from the Osteoarthritis Initiative (OAI). Neither the dataset authors nor the OAI endorse this project.
"""

EXAMPLES = sorted(str(p) for p in EXAMPLES_DIR.glob("*.png"))

# ---------- Interface ----------
with gr.Blocks(title="Knee OA severity grading", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Knee osteoarthritis severity grading\n"
                "Kellgren–Lawrence grade 0–4 from a front-view knee X-ray, "
                "with a heatmap of where the model looked.")
    gr.Markdown("**Research demo only.** Not a medical device and not for diagnosis.")
    history_state = gr.State([])

    with gr.Tabs():
        with gr.Tab("Analyze"):
            with gr.Row():
                with gr.Column(scale=2):
                    inp = gr.Image(type="numpy", image_mode="L", height=320,
                                   label="Knee X-ray (front view, cropped to the joint)")
                    show_cam = gr.Checkbox(value=True, label="Show Grad-CAM heatmap")
                    with gr.Row():
                        run_btn = gr.Button("Grade X-ray", variant="primary")
                        clear_btn = gr.ClearButton(value="Clear")
                    if EXAMPLES:
                        gr.Examples(examples=EXAMPLES, inputs=inp, label="Try an example")
                        gr.Markdown("<small>Examples: dataset labels 0, 2 and 4, shown unmodified. "
                                    "Source: Chen (2018), Mendeley Data, CC BY 4.0.</small>")
                with gr.Column(scale=3):
                    grade_md = gr.Markdown("Upload an X-ray or pick an example, then select **Grade X-ray**.")
                    bars = gr.HTML()
                    with gr.Row():
                        pre_img = gr.Image(label="Preprocessed X-ray", interactive=False, height=240)
                        cam_img = gr.Image(label="Where the model looked (Grad-CAM)", interactive=False, height=240)
            gr.Markdown("#### Recent analyses\nThis browser session only. Nothing is stored on the server.")
            hist_table = gr.Dataframe(headers=["Time", "Predicted grade", "Model probability"],
                                      interactive=False)

        with gr.Tab("Model insights"):
            gr.Markdown(INSIGHTS_MD)
            for fname, label in [("cm_final_test.png", "Test-set confusion matrix"),
                                 ("gradcam_test_grid.png", "Grad-CAM on test images: 4 correct, 2 incorrect per grade"),
                                 ("explanations_compared.png", "Grad-CAM vs occlusion sensitivity"),
                                 ("deletion_test.png", "Deletion test")]:
                if (ASSETS / fname).exists():
                    gr.Image(str(ASSETS / fname), label=label, interactive=False)

        with gr.Tab("About"):
            gr.Markdown(ABOUT_MD)

    gr.Markdown("<small>Knee OA grading demo · Custom CNNs trained from scratch · "
                "For research and education only. Not a medical diagnosis.</small>")

    run_btn.click(analyze, inputs=[inp, show_cam, history_state],
                  outputs=[grade_md, bars, pre_img, cam_img, history_state, hist_table])
    clear_btn.add([inp, pre_img, cam_img, bars])

if __name__ == "__main__":
    demo.launch()

Writing /home/afzal/projects/knee-oa-classification/space/app.py
